# True-State Predictive Readout Control

Tests predictive readout when the exact order-\(k\) Markov state is supplied directly as a one-hot state ID, removing the frozen-representation bottleneck.

Two training targets are evaluated: sampled next-token labels and the known state-conditioned transition distribution. The comparison separates readout/estimation difficulty from representational accessibility.


In [ ]:
# 1. Imports / device / folders
import os, json, math, time, random, hashlib, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path.cwd() / "Markov_Context_Paper" / "Part2_TrueState_Control"
RUNS_DIR = PROJECT_ROOT / "runs"
EXPORTS_DIR = PROJECT_ROOT / "exports"
CSV_DIR = EXPORTS_DIR / "csv"
FIG_DIR = EXPORTS_DIR / "figures"

for p in [PROJECT_ROOT, RUNS_DIR, EXPORTS_DIR, CSV_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
# 2. Configuration matched to Part II
V = 8
KS = [2, 3, 4]
SEEDS = list(range(10))

N_TRAIN = 12_000
N_VAL = 3_000
N_TEST = 6_000

PROBE_BATCH = 512
LINEAR_EPOCHS = 25
MLP_EPOCHS = 30
PATIENCE = 4

LINEAR_LR = 1e-3
MLP_LR = 5e-4
PROBE_WD = 1e-4

RUN_LINEAR = True
RUN_MLP = True
RUN_ORACLE_SOFT = True

SMOKE_TEST = False
if SMOKE_TEST:
    KS = [2]
    SEEDS = [0]
    N_TRAIN, N_VAL, N_TEST = 512, 256, 256
    LINEAR_EPOCHS = 3
    MLP_EPOCHS = 3

print("V:", V, "KS:", KS, "SEEDS:", SEEDS)
print("N:", N_TRAIN, N_VAL, N_TEST)


In [ ]:
# 3. Reproducibility helpers
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def context_to_index(c, V):
    idx = torch.zeros(c.shape[:-1], dtype=torch.long, device=c.device)
    for j in range(c.shape[-1]):
        idx = idx * V + c[..., j]
    return idx


In [ ]:
# 4. Same synthetic Markov source and sampling logic as Part II
def generate_kernel(V, k, seed, device=DEVICE):
    g = torch.Generator(device=device)
    g.manual_seed(seed + 10000019)
    return torch.softmax(
        torch.randn((V**k, V), generator=g, device=device),
        dim=-1
    )

@torch.no_grad()
def generate_dataset(P, V, k, n, seed, batch_size=1024):
    # L=k here because the control receives the true full Markov state.
    g = torch.Generator(device=DEVICE)
    g.manual_seed(seed + 7000009)

    B = min(batch_size, n)
    hist = torch.randint(0, V, (B, k), generator=g, device=DEVICE)

    # Same burn-in length as Part II.
    for _ in range(256):
        probs = P[context_to_index(hist[:, -k:], V)]
        y = torch.multinomial(probs, 1, generator=g).squeeze(1)
        hist = torch.cat([hist[:, 1:], y[:, None]], 1)

    states, ys = [], []
    rem = n

    while rem > 0:
        take = min(B, rem)
        h = hist[:take]
        st = h[:, -k:].clone()
        probs = P[context_to_index(st, V)]
        y = torch.multinomial(probs, 1, generator=g).squeeze(1)

        states.append(st.cpu())
        ys.append(y.cpu())

        probs_all = P[context_to_index(hist[:, -k:], V)]
        ya = torch.multinomial(probs_all, 1, generator=g).squeeze(1)
        hist = torch.cat([hist[:, 1:], ya[:, None]], 1)
        rem -= take

    return torch.cat(states).long(), torch.cat(ys).long()

@torch.no_grad()
def estimate_H_star(P):
    # Kept identical to Part II for direct consistency with its reported baseline.
    p = P.double()
    return float((-(p * torch.log(p.clamp_min(1e-30))).sum(-1)).mean())

@torch.no_grad()
def test_bayes_ce(P, states):
    # Exact CE of the known kernel evaluated on the realized test-state distribution.
    idx = context_to_index(states.to(DEVICE), V)
    p = P[idx].double().clamp_min(1e-30)
    return float((-(p * torch.log(p)).sum(-1)).mean())


## Exact true-state representation

We encode each complete state \(S_t\) as a one-hot vector over the \(V^k\) possible states.  
This is intentionally stronger and cleaner than concatenating token one-hots: an arbitrary random state→transition table is linearly representable from this input.


In [ ]:
# 5. True-state encoding
def state_onehot(states, V):
    k = states.shape[1]
    idx = context_to_index(states.to(DEVICE), V).cpu()
    return F.one_hot(idx, num_classes=V**k).float()


In [ ]:
# 6. Same predictive readout architectures as Part II
class LinearNext(nn.Module):
    def __init__(self, d, V):
        super().__init__()
        self.h = nn.Linear(d, V)

    def forward(self, x):
        return self.h(x)

class MLPNext(nn.Module):
    def __init__(self, d, V):
        super().__init__()
        h = min(1024, max(256, d))
        self.net = nn.Sequential(
            nn.Linear(d, h),
            nn.GELU(),
            nn.Linear(h, V)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
# 7. Training / evaluation
# Hard-label path matches the Part II probe protocol.
def train_probe_hard(model, Xtr, ytr, Xva, yva, epochs, lr):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=PROBE_WD
    )

    tr = DataLoader(
        TensorDataset(Xtr.float(), ytr),
        batch_size=PROBE_BATCH,
        shuffle=True
    )
    va = DataLoader(
        TensorDataset(Xva.float(), yva),
        batch_size=PROBE_BATCH
    )

    best = float("inf")
    best_state = None
    bad = 0

    for ep in range(epochs):
        model.train()
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            z = model(xb)
            loss = F.cross_entropy(z, yb)
            loss.backward()
            opt.step()

        model.eval()
        tot = 0.0
        with torch.no_grad():
            for xb, yb in va:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                z = model(xb)
                loss = F.cross_entropy(z, yb)
                tot += loss.item() * len(xb)

        v = tot / len(Xva)
        if v < best - 1e-5:
            best = v
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    if best_state:
        model.load_state_dict(best_state)

    return model, best

# Diagnostic oracle-soft path: same optimizer and predictor, but targets are
# the known P(.|state), removing stochastic hard-label noise.
def train_probe_soft(model, Xtr, Ptr, Xva, Pva, epochs, lr):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=PROBE_WD
    )

    tr = DataLoader(
        TensorDataset(Xtr.float(), Ptr.float()),
        batch_size=PROBE_BATCH,
        shuffle=True
    )
    va = DataLoader(
        TensorDataset(Xva.float(), Pva.float()),
        batch_size=PROBE_BATCH
    )

    best = float("inf")
    best_state = None
    bad = 0

    for ep in range(epochs):
        model.train()
        for xb, pb in tr:
            xb, pb = xb.to(DEVICE), pb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            z = model(xb)
            logq = F.log_softmax(z, dim=-1)
            loss = -(pb * logq).sum(-1).mean()
            loss.backward()
            opt.step()

        model.eval()
        tot = 0.0
        with torch.no_grad():
            for xb, pb in va:
                xb, pb = xb.to(DEVICE), pb.to(DEVICE)
                z = model(xb)
                logq = F.log_softmax(z, dim=-1)
                loss = -(pb * logq).sum(-1).mean()
                tot += loss.item() * len(xb)

        v = tot / len(Xva)
        if v < best - 1e-5:
            best = v
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    if best_state:
        model.load_state_dict(best_state)

    return model, best

@torch.no_grad()
def eval_hard(model, X, y):
    model.eval()
    dl = DataLoader(
        TensorDataset(X.float(), y),
        batch_size=PROBE_BATCH
    )
    loss = 0.0
    cor = 0
    n = 0
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        z = model(xb)
        l = F.cross_entropy(z, yb)
        loss += l.item() * len(xb)
        cor += (z.argmax(-1) == yb).sum().item()
        n += len(xb)
    return {"next_ce": loss / n, "next_acc": cor / n}

@torch.no_grad()
def eval_soft_ce(model, X, Ptrue):
    model.eval()
    dl = DataLoader(
        TensorDataset(X.float(), Ptrue.float()),
        batch_size=PROBE_BATCH
    )
    loss = 0.0
    n = 0
    for xb, pb in dl:
        xb, pb = xb.to(DEVICE), pb.to(DEVICE)
        z = model(xb)
        logq = F.log_softmax(z, dim=-1)
        l = -(pb * logq).sum(-1).mean()
        loss += l.item() * len(xb)
        n += len(xb)
    return loss / n


In [ ]:
# 8. One k × seed run
def run_one(k, seed):
    run_dir = RUNS_DIR / f"k{k}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    done = run_dir / "DONE"
    out_csv = run_dir / "true_state_readout.csv"

    if done.exists() and out_csv.exists():
        print("SKIP", f"k={k} seed={seed}")
        return pd.read_csv(out_csv)

    set_seed(seed)
    P = generate_kernel(V, k, seed)

    tr_s, tr_y = generate_dataset(P, V, k, N_TRAIN, seed + 101)
    va_s, va_y = generate_dataset(P, V, k, N_VAL, seed + 202)
    te_s, te_y = generate_dataset(P, V, k, N_TEST, seed + 303)

    Xtr = state_onehot(tr_s, V)
    Xva = state_onehot(va_s, V)
    Xte = state_onehot(te_s, V)
    d = Xtr.shape[1]

    # True transition distributions for the same realized states.
    with torch.no_grad():
        Ptr = P[context_to_index(tr_s.to(DEVICE), V)].cpu().float()
        Pva = P[context_to_index(va_s.to(DEVICE), V)].cpu().float()
        Pte = P[context_to_index(te_s.to(DEVICE), V)].cpu().float()

    H_star_uniform = estimate_H_star(P)
    H_star_test = test_bayes_ce(P, te_s)

    rows = []

    specs = []
    if RUN_LINEAR:
        specs.append(("linear", LinearNext, LINEAR_EPOCHS, LINEAR_LR))
    if RUN_MLP:
        specs.append(("mlp", MLPNext, MLP_EPOCHS, MLP_LR))

    for probe_type, cls, epochs, lr in specs:
        # Primary: same hard-label task as Part II.
        set_seed(1_000_000 + 1000*k + 10*seed + (0 if probe_type == "linear" else 1))
        model = cls(d, V)
        model, best_val = train_probe_hard(
            model, Xtr, tr_y, Xva, va_y, epochs, lr
        )
        hard = eval_hard(model, Xte, te_y)
        soft_ce_after_hard = eval_soft_ce(model, Xte, Pte)

        rows.append({
            "k": k,
            "seed": seed,
            "V": V,
            "state_dim": d,
            "probe_type": probe_type,
            "training_target": "hard_sampled_next_token",
            "N_TRAIN": N_TRAIN,
            "N_VAL": N_VAL,
            "N_TEST": N_TEST,
            "H_star_uniform": H_star_uniform,
            "H_star_test": H_star_test,
            "best_val_ce": best_val,
            "next_ce": hard["next_ce"],
            "next_acc": hard["next_acc"],
            "soft_ce_on_true_P": soft_ce_after_hard,
            "excess_over_Hstar_test": soft_ce_after_hard - H_star_test,
        })

        del model

        # Diagnostic: oracle-soft targets isolate predictor capacity.
        if RUN_ORACLE_SOFT:
            set_seed(2_000_000 + 1000*k + 10*seed + (0 if probe_type == "linear" else 1))
            model = cls(d, V)
            model, best_val_soft = train_probe_soft(
                model, Xtr, Ptr, Xva, Pva, epochs, lr
            )
            hard_eval = eval_hard(model, Xte, te_y)
            soft_ce = eval_soft_ce(model, Xte, Pte)

            rows.append({
                "k": k,
                "seed": seed,
                "V": V,
                "state_dim": d,
                "probe_type": probe_type,
                "training_target": "oracle_transition_distribution",
                "N_TRAIN": N_TRAIN,
                "N_VAL": N_VAL,
                "N_TEST": N_TEST,
                "H_star_uniform": H_star_uniform,
                "H_star_test": H_star_test,
                "best_val_ce": best_val_soft,
                "next_ce": hard_eval["next_ce"],
                "next_acc": hard_eval["next_acc"],
                "soft_ce_on_true_P": soft_ce,
                "excess_over_Hstar_test": soft_ce - H_star_test,
            })

            del model

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)

    save_json(
        run_dir / "config.json",
        {
            "V": V,
            "k": k,
            "seed": seed,
            "state_encoding": "one_hot_exact_state_id",
            "state_dim": int(d),
            "N_TRAIN": N_TRAIN,
            "N_VAL": N_VAL,
            "N_TEST": N_TEST,
            "PROBE_BATCH": PROBE_BATCH,
            "LINEAR_EPOCHS": LINEAR_EPOCHS,
            "MLP_EPOCHS": MLP_EPOCHS,
            "PATIENCE": PATIENCE,
            "LINEAR_LR": LINEAR_LR,
            "MLP_LR": MLP_LR,
            "PROBE_WD": PROBE_WD,
        }
    )
    done.write_text("completed\n")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return df


In [ ]:
# 9. Sanity checks
for k in KS:
    P = generate_kernel(V, k, 123)
    st, y = generate_dataset(P, V, k, 64, 777)
    X = state_onehot(st, V)

    assert st.shape == (64, k)
    assert y.shape == (64,)
    assert X.shape == (64, V**k)
    assert torch.allclose(X.sum(1), torch.ones(64))
    assert torch.isfinite(X).all()

    # Critical representability check:
    # exact-state one-hot has a unique coordinate for each full state.
    ids = context_to_index(st.to(DEVICE), V).cpu()
    assert torch.equal(X.argmax(1), ids)

    print(f"SANITY PASS k={k} state_dim={V**k}")

print("ALL SANITY CHECKS PASSED")


In [ ]:
# 10. Run all 3 k values × 10 seeds
START = time.time()
all_rows = []

for k in KS:
    for seed in SEEDS:
        print("=" * 70)
        print("RUN", f"k={k} seed={seed}")
        all_rows.append(run_one(k, seed))

results = pd.concat(all_rows, ignore_index=True)
raw_path = CSV_DIR / "true_state_predictive_readout_raw.csv"
results.to_csv(raw_path, index=False)

print("Rows:", len(results))
print("Elapsed minutes:", round((time.time() - START) / 60, 2))
print("RAW:", raw_path)
display(results.head(20))


In [ ]:
# 11. Publication-style summary
summary = (
    results
    .groupby(["k", "probe_type", "training_target"], as_index=False)
    .agg(
        state_dim=("state_dim", "first"),
        H_star_test_mean=("H_star_test", "mean"),
        H_star_test_std=("H_star_test", "std"),
        next_ce_mean=("next_ce", "mean"),
        next_ce_std=("next_ce", "std"),
        next_acc_mean=("next_acc", "mean"),
        soft_ce_trueP_mean=("soft_ce_on_true_P", "mean"),
        soft_ce_trueP_std=("soft_ce_on_true_P", "std"),
        excess_Hstar_mean=("excess_over_Hstar_test", "mean"),
        excess_Hstar_std=("excess_over_Hstar_test", "std"),
        n=("seed", "count"),
    )
)

summary_path = CSV_DIR / "true_state_predictive_readout_summary.csv"
summary.to_csv(summary_path, index=False)

print("SUMMARY:", summary_path)
display(summary)


In [ ]:
# 12. Figure — excess CE above the true-state Bayes floor
plt.figure(figsize=(8, 5.5))

plot_df = summary.copy()
for (probe_type, target), d in plot_df.groupby(["probe_type", "training_target"]):
    label = f"{probe_type} | {target}"
    plt.errorbar(
        d["k"],
        d["excess_Hstar_mean"],
        yerr=d["excess_Hstar_std"].fillna(0),
        marker="o",
        capsize=3,
        label=label,
    )

plt.axhline(0, linewidth=1)
plt.xlabel("Markov order k")
plt.ylabel("True-state readout CE - Bayes CE (nats)")
plt.title("True-state predictive-readout control")
plt.xticks(KS)
plt.legend()
plt.tight_layout()

png = FIG_DIR / "true_state_predictive_readout_control.png"
pdf = FIG_DIR / "true_state_predictive_readout_control.pdf"
plt.savefig(png, dpi=300)
plt.savefig(pdf)
plt.show()

print("FIGURES:", png, pdf)


In [ ]:
# 13. Integrity audit
expected_runs = len(KS) * len(SEEDS)
expected_rows_per_run = (int(RUN_LINEAR) + int(RUN_MLP)) * (1 + int(RUN_ORACLE_SOFT))
expected_rows = expected_runs * expected_rows_per_run

done_dirs = [p.parent for p in RUNS_DIR.rglob("DONE")]

num_cols = [
    "H_star_uniform",
    "H_star_test",
    "best_val_ce",
    "next_ce",
    "next_acc",
    "soft_ce_on_true_P",
    "excess_over_Hstar_test",
]

audit = {
    "expected_runs": expected_runs,
    "done_runs": len(done_dirs),
    "expected_rows": expected_rows,
    "actual_rows": len(results),
    "missing_result_files": sum(
        not (d / "true_state_readout.csv").exists()
        for d in done_dirs
    ),
    "nonfinite_rows": int(
        (~np.isfinite(results[num_cols].to_numpy(float))).any(1).sum()
    ),
    "duplicate_rows": int(
        results.duplicated(
            ["k", "seed", "probe_type", "training_target"]
        ).sum()
    ),
    "all_seed_counts_10": bool(
        (summary["n"] == len(SEEDS)).all()
    ),
}

audit_path = EXPORTS_DIR / "true_state_integrity_audit.json"
save_json(audit_path, audit)

print(audit)
assert audit["done_runs"] == expected_runs
assert audit["actual_rows"] == expected_rows
assert audit["missing_result_files"] == 0
assert audit["nonfinite_rows"] == 0
assert audit["duplicate_rows"] == 0
assert audit["all_seed_counts_10"]

print("INTEGRITY AUDIT PASSED")


In [ ]:
# 14. Package control outputs
output = Path.cwd() / "Part2_TrueState_PredictiveReadout_Control_RESULTS.zip"
allowed = {".csv", ".json", ".png", ".pdf", ".txt"}

with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as z:
    for f in PROJECT_ROOT.rglob("*"):
        if f.is_file() and f.suffix.lower() in allowed:
            z.write(f, f.relative_to(PROJECT_ROOT))

print("ZIP:", output)
print("MB:", round(output.stat().st_size / 1024**2, 2))
